### Goal
Apply the grid search over the LSTM and obtains the best hyperparameters

#### 1. Setup

In [ ]:
# Importing packages
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader
from pa_model import Dataset, SWDataset
from pa_model import DATA_RAW, DATA_PROCESSED, RESULTS
from metrics.metrics import RMSE, EVM, RSQR, NMSE, ACEPR
from plots import (plot_AM_AM, plot_AM_PM, plot_PM_PM, plot_constellation, plot_PSD,
                   plot_comparison, COLOR_DATASET, COLOR_LSTM, COLOR_ERROR)
from lstm import PaLSTM, GridSearchLSTM
import matplotlib.pyplot as plt

seed = 42
generator = torch.Generator().manual_seed(seed)

In [2]:
# Choosing the dataset: "pa_proprio" (dadosIniciais/expData) or "APA_200MHz" (OpenDPD, see data/raw/APA_200MHz/SOURCE.md)
DATASET = "APA_200MHz"

# Loading Dataset
if DATASET == "pa_proprio":
    path_initial_data = f"{DATA_RAW}/dadosIniciais.csv"
    path_experiment_data = f"{DATA_PROCESSED}/expData.csv"

    if Path(path_experiment_data).exists():
        ds = Dataset.from_csv(path_experiment_data)
    else:
        df = pd.read_csv(path_initial_data)
        df = df.loc[df.index.repeat(10)].reset_index(drop=True)
        df = df.sample(frac=1,random_state=42).reset_index(drop=True)
        df.to_csv(path_experiment_data,index=False)
        ds = Dataset.from_csv(path_experiment_data)

In [ ]:
# Splitting Data into train, val and test Datasets
if DATASET == "APA_200MHz":
    train, val, test = Dataset.from_split_csv(DATA_RAW / "APA_200MHz")   # official split, temporal order
else:
    train, val, test = ds.split()

print(DATASET, "->", len(train), len(val), len(test))

In [ ]:
 # Defining the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: ", device)

In [ ]:
# GridSearch Hyperparameters# Parametros do gridsearch
param_grid = {
    "window_size": [10, 20, 30],
    "hidden_size": [2, 4, 8, 16],
    "num_layers": [1, 2, 3, 4, 5],
    "learning_rate": [0.001],
    "dropout": [0.0, 0.5],
    "batch_size": [128, 512, 1024],
}

#### 2. Running Grid Search

Backup of the previous results (baseline) before running the new experiment.

- `BACKUP_DIR`: copy of `results/` as it was before the optimized `SWDataset`/`TrainerLSTM` (2026-11-05)
- `RUN_DIR`: the new grid search writes here. It must NOT be `results/` itself: `ResultsStorage` would skip every combination whose history already exists and would compare against the old best model.

In [ ]:
# Backup of the previous results
import shutil

BACKUP_DIR = RESULTS.parent / "results_backup_2026-11-05"
# Each PA has its own results folder (ResultsStorage must never mix them)
RUN_DIR = RESULTS / ("APA_200MHz" if DATASET == "APA_200MHz" else "v2")

if BACKUP_DIR.exists():
    print(f"Backup already exists, nothing copied: {BACKUP_DIR}")
else:
    shutil.copytree(RESULTS, BACKUP_DIR, ignore=shutil.ignore_patterns("v2", "APA_200MHz"))
    print(f"Backup created: {BACKUP_DIR}")

print("Backup contents:", sorted(p.name for p in BACKUP_DIR.iterdir()))
print("New run directory:", RUN_DIR)

In [ ]:
# Creating LSTM Grid Search Object
gsearch = GridSearchLSTM(train, val, param_grid, device)

In [ ]:
# Running Grid Search
best_metric, best_model, best_params = gsearch.run(RUN_DIR)

In [ ]:
# Saving Results
dict_results = {"best_evm": best_metric}
dict_results.update(best_params)

with open(RUN_DIR/"best_result.json", "w") as f:
    json.dump(dict_results, f)

In [ ]:
# Saving Model
torch.save(best_model.state_dict(), RUN_DIR/"best_model.pkl")

#### 3. Analysing Results

In [ ]:
# Reading best hyperparameters
# APA_200MHz: this run; pa_proprio: still the old model (until v2 is compared with the backup)
ANALYSIS_DIR = RUN_DIR if DATASET == "APA_200MHz" else RESULTS

with open(ANALYSIS_DIR/"best_params.json", "r") as f:
    data = json.load(f)

print(f"{type(data)}")
print(f"{data}")
params = data['params']

In [ ]:
# Building Test Sliding Window Dataset
train_norm, scaler_x, scaler_y = train.normalize()
test_norm, _, _ = test.normalize(scaler_x, scaler_y)
train_dl = DataLoader(SWDataset(train_norm, params['ws']), batch_size=params['bs'], shuffle=True, generator=generator)
test_dl = DataLoader(SWDataset(test_norm, params['ws']), batch_size=params['bs'], shuffle=False, generator=generator)

In [ ]:
# Building Model
model = PaLSTM(2,params['hs'],params['nl'],params['dr'])
model.load_state_dict(torch.load(ANALYSIS_DIR/"best_model.pt", weights_only=True))
model.to(device)
model.eval()

In [ ]:
# Applying the model over the test dataset
preds, targets = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        xb = (xb.to(torch.float32)).to(device)
        out = model(xb)
        preds.append(out.cpu().numpy())
        targets.append(yb.numpy())

y_pred = np.array(scaler_y.inverse_transform(np.concatenate(preds)))
y_true = np.array(scaler_y.inverse_transform(np.concatenate(targets)))

y_pred = y_pred[:, 0] + 1j * y_pred[:, 1]
y_true = y_true[:, 0] + 1j * y_true[:, 1]

# Input aligned with the predictions: each window drops its first ws-1 samples
x = test.x[params['ws']-1:]

In [ ]:
# Metrics over the test dataset
# Spectral parameters: only APA_200MHz is in temporal order (PSD/ACEPR make no sense on shuffled data)
if DATASET == "APA_200MHz":
    spec = json.load(open(DATA_RAW / "APA_200MHz" / "spec.json"))
    FS, BW = spec["input_signal_fs"], spec["bw_main_ch"]          # 983.04 MHz, 200 MHz
else:
    FS = BW = None

results = {
    "EVM (%)":   EVM().compute(y_true, y_pred),
    "RMSE":      RMSE().compute(y_true, y_pred),
    "R²":        RSQR().compute(y_true, y_pred),
    "NMSE (dB)": NMSE().compute(y_true, y_pred),
}
if FS is not None:
    results["ACEPR (dB)"] = ACEPR(FS, BW).compute(y_true, y_pred)

for name, value in results.items():
    print(f"{name:<11}{value:12.6f}")

In [ ]:
plot_AM_AM(x, y_pred, color=COLOR_LSTM,    title="AM/AM Graph [LSTM Forecast over Test Dataset]")
plot_AM_AM(x, y_true, color=COLOR_DATASET, title="AM/AM Graph [Test Dataset]")

In [ ]:
plot_AM_PM(x, y_pred, color=COLOR_LSTM,    title="AM/PM Graph [LSTM Forecast over Test Dataset]")
plot_AM_PM(x, y_true, color=COLOR_DATASET, title="AM/PM Graph [Test Dataset]")

In [ ]:
plot_PM_PM(x, y_pred, color=COLOR_LSTM,    title="PM/PM Graph [LSTM Forecast over Test Dataset]")
plot_PM_PM(x, y_true, color=COLOR_DATASET, title="PM/PM Graph [Test Dataset]")

In [ ]:
# Constellation (I x Q in the time domain: OFDM looks like a circular cloud)
plot_constellation(y_pred, color=COLOR_LSTM,    title="Constellation [LSTM Forecast over Test Dataset]")
plot_constellation(y_true, color=COLOR_DATASET, title="Constellation [Test Dataset]")

In [ ]:
if FS is not None:
    plot_PSD({"PA output (measured)": y_true, "LSTM prediction": y_pred, "Error": y_true - y_pred}, FS, BW,
             colors={"PA output (measured)": COLOR_DATASET, "LSTM prediction": COLOR_LSTM, "Error": COLOR_ERROR},
             title="PSD [Test Dataset]  (dashed: main / adjacent channel edges)")
else:
    print("PSD skipped: pa_proprio is shuffled (no temporal order)")

#### 4. Comparison: Dataset vs LSTM
All the graphs above in a single grid, with both series overlaid (dataset in blue, LSTM in orange).

In [ ]:
plot_comparison(x, y_true, y_pred, fs=FS, bw=BW, title=f"Dataset vs LSTM [{DATASET}, Test Dataset]")